### Impotacion y preprocesado de datos

In [1]:
from importacion_preprocesado import descarga_y_carga_de_datos, preprocesamiento


tamany_img = (128,128) # tamaño reducido para colab, porque si no supera la ram
X, y = descarga_y_carga_de_datos(target_size=tamany_img)

X_train, X_val, X_test, y_train, y_val, y_test = preprocesamiento(X, y)

Dataset ya existe, solo se van a cargar las imágenes.
X shape: (4217, 128, 128, 3) y shape: (4217,)


### Modelo

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tensorflow.keras.applications import ResNet50, VGG16
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.optimizers import Adam, SGD
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.preprocessing.image import ImageDataGenerator


# ========== OPCIÓN 1: ResNet50 (RECOMENDADO) ==========


# Cargar ResNet50 pre-entrenado (sin capas finales)
restnet = ResNet50(
  weights='imagenet',  # Pre-entrenado en ImageNet
  input_shape=(128,128,3),
  include_top=False  # Sin capa de salida (la agregaremos nosotros)
)

for layers in restnet.layers:
    layers.trainable = False


# restnet.summary()

In [3]:
# CREAR modelo personalizado
model = Sequential()

# Agregar capas personalizadas para nuestro problema
model.add(restnet)


model.add(GlobalAveragePooling2D())
model.add(Dense(512, activation='relu'))
model.add(Dropout(0.4))
model.add(Dense(256, activation='relu'))
model.add(Dropout(0.3))
model.add(Dense(4, activation='softmax'))

# Compilar
optimizer = Adam(learning_rate=0.0001)
model.compile(
  optimizer=optimizer,
  loss='categorical_crossentropy',
  metrics=['accuracy']
)



In [ ]:
hist = model.fit(X_train, y_train, validation_data= (X_val, y_val), epochs= 20)

Epoch 1/20
80/80 ━━━━━━━━━━━━━━━━━━━━ 110s 1s/step - accuracy: 0.2487 - loss: 1.4801 - val_accuracy: 0.3780 - val_loss: 1.3748
Epoch 2/20
80/80 ━━━━━━━━━━━━━━━━━━━━ 84s 1s/step - accuracy: 0.2653 - loss: 1.4299 - val_accuracy: 0.3081 - val_loss: 1.3731
Epoch 3/20
80/80 ━━━━━━━━━━━━━━━━━━━━ 82s 1s/step - accuracy: 0.2728 - loss: 1.4090 - val_accuracy: 0.3069 - val_loss: 1.3613
Epoch 4/20
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 760ms/step - accuracy: 0.2803 - loss: 1.3839

In [ ]:
import matplotlib.pyplot as plt
# Visualizacion evolucion loss durante el entrenamiento
plt.plot(hist.history['loss'],label="loss")
plt.plot(hist.history['val_loss'],label="val_loss")
plt.legend()
plt.show()

# Visualizacion de accuracy durante el entrenamiento
plt.plot(hist.history["accuracy"], label= "accuracy")
plt.plot(hist.history["val_accuracy"], label ="val_accuracy")
plt.legend()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

datasets = {
    "Train": (X_train, y_train),
    "Val": (X_val, y_val),
    "Test": (X_test, y_test)
}

def evaluate(model, X, y):
    #Convertir a enteros
    y_true = np.argmax(y, axis=1)

    y_pred_probs = model.predict(X)
    y_pred = np.argmax(y_pred_probs, axis=1)

    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average="weighted")
    return acc, f1


def crear_df_metricas(model, datasets):
    results = {}
    for split_name, (X, y) in datasets.items():
        acc, f1 = evaluate(model, X, y)
        results[split_name] = [acc, f1]
    
    df = pd.DataFrame(results, index=["Accuracy", "F1"])
    return df

In [ ]:
df = crear_df_metricas(model, datasets)
print(df.round(3))

In [ ]:
def plot_barra(metrica, titulo, color):

    plt.figure(figsize=(6,4))
    bars = plt.bar(metrica.index, metrica.values, color=color)
    plt.bar_label(bars)
    plt.title(f"{titulo}: Train vs Validation vs Test")
    plt.ylabel(titulo)
    plt.ylim(0,1)
    plt.grid(axis='y', linestyle='--')
    plt.show()


def plots_metricas(df):
    #Extraemos la fila
    accuracy = df.loc["Accuracy"]
    f1 = df.loc["F1"]

    #Grafico de Accuracy
    plot_barra(accuracy, titulo = "Accuracy", color="blue")

    #Grafico de 
    plot_barra(f1, titulo ="F1", color = "red")

In [ ]:
plots_metricas(df)

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

y_true = np.argmax(y_test, axis=1)
y_pred = np.argmax(model.predict(X_test), axis=1)

cm = confusion_matrix(y_true, y_pred)
ConfusionMatrixDisplay(cm,  
                    display_labels= ["cataract", "diabetic_retinopathy", "glaucoma", "normal"]
                    ).plot(cmap="Blues", xticks_rotation= 45.0)

In [ ]:
def per_class_table(model, datasets):
    # class_names: lista con nombres de las clases en el orden de los índices (opcional)
    class_names = ["cataract", "diabetic_retinopathy", "glaucoma", "normal"]  

    splits = list(datasets.keys())
    # identificamos número de clases desde los datos si no dan nombres
    sample_X, sample_y = next(iter(datasets.values()))
    n_classes = np.argmax(sample_y, axis=1).max() + 1

    # DataFrame con index = clases y columnas por split/metric
    cols = []
    for s in splits:
        cols += [f"{s}_Accuracy", f"{s}_F1"]
    df = pd.DataFrame(index=class_names, columns=cols, dtype=float)

    for split in splits:
        X, y = datasets[split]
        y_true = np.argmax(y, axis=1)
        y_pred = np.argmax(model.predict(X), axis=1)

        # precision, recall, f1 por clase
        precision, recall, f1, support = precision_recall_fscore_support(
            y_true, y_pred, labels=np.arange(n_classes), zero_division=0
        )
        # Aquí definimos "Accuracy" por clase como recall (TP / nº verdaderos de la clase)
        for i, cls in enumerate(class_names):
            df.loc[cls, f"{split}_Accuracy"] = recall[i]
            df.loc[cls, f"{split}_F1"] = f1[i]

    return df

In [ ]:
df_per_class = per_class_table(model, datasets)
print(df_per_class.round(3))

Resultado resnet:
          Train    Val   Test
Accuracy  0.484  0.468  0.469
F1        0.445  0.431  0.439

resultDO VGG:
          Train    Val   Test
Accuracy  0.811  0.808  0.807
F1        0.811  0.809  0.807
